In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys
import pandas as pd

# Auto-find project root containing src/__init__.py (robust)
cwd = Path.cwd().resolve()
project_root = None
for p in [cwd, *cwd.parents]:
    if (p / "src" / "__init__.py").exists():
        project_root = p
        break
if project_root is None:
    raise FileNotFoundError("Could not find project root containing src/__init__.py")

sys.path.insert(0, str(project_root))

from src.governance import (
    PrivacyPaths,
    load_privacy_inputs,
    normalize_pii_inventory,
    build_pii_presence_table,
    assert_no_direct_pii_in_analysis,
    heuristic_suspicious_columns,
    safe_preview_curated,
    demo_pseudonymise,
    extract_processing_timestamp_rows,
)

In [3]:


# Define dataset paths (project-standard locations)
paths = PrivacyPaths(
    pii_inventory=Path("data/quality/pii_inventory.csv"),
    applications_analysis=Path("data/curated/applications_analysis.csv"),
    applications_curated_full=Path("data/curated/applications_curated_full.csv"),
    dq_postclean=Path("data/quality/reports/post/data_quality_report_postclean.csv"),
)

# Load the core datasets (PII + analysis + curated)
# We do NOT load dq_post here yet, because it may not exist.
pii = pd.read_csv(paths.pii_inventory)
analysis = pd.read_csv(paths.applications_analysis)
curated = pd.read_csv(paths.applications_curated_full)

print("PII inventory:", pii.shape)
print("Analysis dataset:", analysis.shape)
print("Curated full dataset:", curated.shape)

# Ensure the post-clean DQ report exists; if not, generate it.
if not paths.dq_postclean.exists():
    print("DQ post-clean report not found. Generating it now...")
    write_data_quality_report_postclean(
        applications_curated_full_path=paths.applications_curated_full,
        output_path=paths.dq_postclean,
    )

dq_post = pd.read_csv(paths.dq_postclean)
print("DQ post-clean report:", dq_post.shape)


FileNotFoundError: [Errno 2] No such file or directory: 'data\\quality\\pii_inventory.csv'

In [ ]:
# Normalize inventory and extract Direct PII vs Quasi-identifiers/proxies
inv, direct_fields, quasi_fields, inv_map = normalize_pii_inventory(pii)

print(f"Direct PII fields (n={len(direct_fields)}):")
print(direct_fields)

print(f"\nQuasi-identifiers / proxy candidates (n={len(quasi_fields)}):")
print(quasi_fields)

# Optional: show the inferred mapping (good for debugging once, then you can remove it)
print("\nInventory schema mapping:", inv_map)

NameError: name 'pii' is not defined

In [ ]:
# Build a governance-friendly "presence by layer" table
presence = build_pii_presence_table(
    inv=inv,
    inv_map=inv_map,
    analysis_cols=set(map(str, analysis.columns)),
    curated_cols=set(map(str, curated.columns)),
)

display(presence)

NameError: name 'inv' is not defined

In [ ]:
# Hard control evidence: Direct PII must NOT exist in the modelling dataset
assert_no_direct_pii_in_analysis(direct_fields, analysis)
print("✅ Data minimisation control PASSED: No Direct PII columns in applications_analysis.csv")

NameError: name 'direct_fields' is not defined